In [2]:
#imports: do not change them
import torch
import matplotlib.pyplot as plt
import numpy as np
import argparse
import pickle
import os
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import torchvision
import random
from tqdm import tqdm
from torch.utils.data import random_split
from torch.autograd import Variable
import cv2
import torchvision.utils as utils

## Please DONOT remove these lines.
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(0)

### Question 1: MultiHead-Attention
Given a set of query, key, value vectors, calculate the attention value using multihead attention.

choose the closest option to the maximum value of the attention vector


1.   0.5590
2.   0.7851
3.   0.2312
4.   0.9452



In [ ]:
## Please DO NOT change these values.
#This is the config that will be used for implementing the Multi-head attention
config = {
    'QI_DIM': 64,
    'VI_DIM': 64,
    'KI_DIM': 64,
    'QO_DIM': 32,
    'VO_DIM': 32,
    'OP_DIM': 32,
    'NUM_HEADS': 8,
}

In [ ]:
#A class for implementing the Multi-head Attention
class MultiHeadAttention(nn.Module):

    def __init__(self, config):
        super(MultiHeadAttention, self).__init__()

        ### YOUR CODE STARTS HERE ###

        #set the op dimension, number of heads based on the value given in config
        self.op = config['OP_DIM']
        self.num_heads = config['NUM_HEADS']

        #set the query dimension as QO_DIM from the config and value dimension as VO_DIM
        self.qo_dim = config['QO_DIM']
        self.vo_dim = config['VO_DIM']

        #set the respective dimensions
        self.qi_dim = config['QI_DIM']
        self.ki_dim = config['KI_DIM']
        self.vi_dim = config['VI_DIM']
        self.q = nn.Linear(self.qi_dim, self.qo_dim)
        self.k = nn.Linear(self.ki_dim, self.qo_dim)
        self.v = nn.Linear(self.vi_dim, self.vo_dim)
        self.OTrans = nn.Linear(self.qo_dim, self.qo_dim)

        #set the scaling factor as follows:
        #scale = sqrt(QO_DIM/NUM_HEADS)
        self.scale = np.sqrt(self.qo_dim/self.num_heads)

        ### YOUR CODE ENDS HERE ###



    def forward(self, q_inp, k_inp, v_inp, ret_atn=False):

        #making sure that shapes are similar across dim=0
        assert q_inp.shape[0] == k_inp.shape[0]

        #setting batch size
        batch_size = q_inp.shape[0]
        seq_length = q_inp.shape[1]

        ### YOUR CODE STARTS HERE ###
        #pass q_inp, k_inp, v_inp through the linear layers defined in the
        #init function

        Q = self.q(q_inp)
        K = self.k(k_inp)
        V = self.v(v_inp)
        print(Q.shape, K.shape, V.shape)
        #reshape the Q, K, V tensors to (batch_size, num_heads, -1, self.query_dim/num_heads)
        #hint: you can also do the same by the view() of pytorch
        Q = Q.view(batch_size, self.num_heads, -1, self.qo_dim//self.num_heads)
        K = K.view(batch_size, self.num_heads, -1, self.qo_dim//self.num_heads)
        V = V.view(batch_size, self.num_heads, -1, self.qo_dim//self.num_heads)
        print(Q.shape, K.shape, V.shape)


        # define energy as matrix product of Q and K divided by self.scale value
        #hint: use torch.matmul() for this and do not forget to reshape K vector as (batch_size, num_heads, self.query_dim/num_heads, -1)
        #this should be done in order to carry out the matrix multiplication
        energy = torch.matmul(Q, K.permute(0,1,3,2)) / self.scale
        print(energy.shape)
        #define attention as softmax of energy vector across the last dimension
        attention = F.softmax(energy, dim = -1)
        print(attention.shape)
        ### YOUR CODE ENDS HERE ###

        #multiplying the attention with value tensor
        attended = torch.matmul(attention, V).permute(0, 2, 1, 3).contiguous()

        #passing the attended tensor through OTrans linear layer
        out = self.OTrans(attended.view(batch_size, seq_length, -1))

        if ret_atn:
            return out, attention

        else:
            return out

In [ ]:
#From ChatGPT
class MultiHeadAttention(nn.Module):

    def __init__(self, config):
        super(MultiHeadAttention, self).__init__()

        self.num_heads = config['NUM_HEADS']
        self.query_dim = config['QO_DIM']
        self.value_dim = config['VO_DIM']

        # Ensure QO_DIM is divisible by NUM_HEADS
        assert self.query_dim % self.num_heads == 0

        # Dimensions for queries, keys, and values after splitting
        self.head_dim = self.query_dim // self.num_heads

        # Linear layers for Q, K, V
        self.linear_q = nn.Linear(config['QI_DIM'], self.query_dim)
        self.linear_k = nn.Linear(config['KI_DIM'], self.query_dim)
        self.linear_v = nn.Linear(config['VI_DIM'], self.value_dim)

        # Output linear layer
        self.linear_o = nn.Linear(self.value_dim, self.query_dim)

        # Scaling factor
        self.scale = torch.sqrt(torch.FloatTensor([self.head_dim]))

    def forward(self, q_inp, k_inp, v_inp, ret_atn=False):
        assert q_inp.shape[0] == k_inp.shape[0] == v_inp.shape[0]

        batch_size = q_inp.shape[0]
        seq_length = q_inp.shape[1]

        # Linear transformations
        Q = self.linear_q(q_inp)
        K = self.linear_k(k_inp)
        V = self.linear_v(v_inp)

        # Reshape Q, K, V to (batch_size, num_heads, seq_length, head_dim)
        Q = Q.view(batch_size, seq_length, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch_size, seq_length, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch_size, seq_length, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        # Calculate energy
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale

        # Apply attention using softmax
        attention = F.softmax(energy, dim=-1)

        # Apply attention to the value vectors
        attended = torch.matmul(attention, V)

        # Rearrange to (batch_size, seq_length, num_heads, head_dim)
        attended = attended.permute(0, 2, 1, 3).contiguous()

        # Concatenate heads and apply final linear transformation
        attended = attended.view(batch_size, seq_length, self.query_dim)
        out = self.linear_o(attended)

        if ret_atn:
            return out, attention
        else:
            return out


In [ ]:
## Please DONOT remove these lines.
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(0)

q = torch.rand(4, 2, 64)
k = torch.rand(4, 2, 64)
v = torch.rand(4, 2, 64)
## Please DONOT remove these lines.

### YOUR CODE STARTS HERE ###

#initialise an instance of MultiHeadAttention class
d = MultiHeadAttention(config)

#pass q, k, v values through the instance
x, y = d(q,k,v, True)

#print the max value of the attention and report your answer
print(torch.max(y))

### YOUR CODE ENDS HERE ###


tensor(0.5590, grad_fn=<MaxBackward1>)


### Question 2: Learning to pay attention
For questions 2 & 3, we will implement the paper "Learning to pay Attention", published at ICLR2018. The paper focuses on Trainable Soft Visual Attention in CNNs for image classification task. Follow the steps given in the upcoming code blocks to answer question 2.

What is the range of the mean value obtained?


1.   0.01 - 0.1
2.   0.1 - 0.3
3.   0.5 - 0.8
4.   1 - 1.5



In [2]:
#defining a block that has conv->batchnorm->relu
class ConvBlock(nn.Module):
    def __init__(self, in_features, out_features, num_conv, pool=False):
        super(ConvBlock, self).__init__()
        features = [in_features] + [out_features for i in range(num_conv)]

        #appending all the operations in the list and then converting them to sequential set of operations
        layers = []
        for i in range(len(features)-1):
            layers.append(nn.Conv2d(in_channels=features[i], out_channels=features[i+1], kernel_size=3, padding=1, bias=True))
            layers.append(nn.BatchNorm2d(num_features=features[i+1], affine=True, track_running_stats=True))
            layers.append(nn.ReLU())
            if pool:
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2, padding=0))
        self.op = nn.Sequential(*layers)
    def forward(self, x):
        return self.op(x)

#defining a projector block which is nothing but a conv layer that is used to transform the channel dimension
class ProjectorBlock(nn.Module):
    def __init__(self, in_features, out_features):
        super(ProjectorBlock, self).__init__()
        self.op = nn.Conv2d(in_channels=in_features, out_channels=out_features, kernel_size=1, padding=0, bias=False)
    def forward(self, inputs):
        return self.op(inputs)


#defining a linear attention block as mentioned in the paper
class LinearAttentionBlock(nn.Module):
    def __init__(self, in_features, normalize_attn=True):
        super(LinearAttentionBlock, self).__init__()

        ### YOUR CODE STARTS HERE ###
        #set self.mormalize_attn as the normalize_attn argument
        self.normalize_attn = normalize_attn

        #define self.op as a conv layer with in_channels=in_features, out_channels=1, kernel_size=1, padding=0 and with bias=False
        self.op =

        ### YOUR CODE ENDS HERE ###


    def forward(self, l, g):
        ### YOUR CODE STARTS HERE ###

        #set N, C, W, H using the size of the 'l' tensor
        #where N = number of batches, C = channels, H = height of the tensor, W = width of the tensor
        N, C, W, H =

        #add l and g as mentioned in the paper and pass it through the self.op layer
        c =  # batch_sizex1xWxH

        #pass the output of the self.op layer through sigmoid activation function
        a =

        #make the size of a same as l
        #hint: you can do this by the tensor.expand_as() method of PyTorch

        #multiple a and l using torch.mul()
        g =

        ### YOUR CODE ENDS HERE ###

        #using adaptive avg pooling as the final operation
        g = F.adaptive_avg_pool2d(g, (1,1)).view(N,C)

        return c.view(N,1,W,H), g


SyntaxError: invalid syntax (<ipython-input-2-36755c46244a>, line 38)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LinearAttentionBlock(nn.Module):
    def __init__(self, in_features, normalize_attn=True):
        super(LinearAttentionBlock, self).__init__()

        self.normalize_attn = normalize_attn

        # Define the 1x1 convolutional layer for attention computation
        self.op = nn.Conv2d(in_channels=in_features, out_channels=1, kernel_size=1, padding=0, bias=False)

    def forward(self, l, g):
        N, C, W, H = l.size()

        # Concatenate l and g along the channel dimension
        # Note: Ensure g is reshaped to match the channel dimensions of l
        c = torch.cat((l, g.expand(N, 1, W, H)), dim=1)  # N x (C+1) x W x H

        # Apply 1x1 convolution to get attention scores
        c = self.op(c)  # N x 1 x W x H

        # Apply sigmoid activation to get attention weights
        a = torch.sigmoid(c)  # N x 1 x W x H

        # Normalize attention weights if specified
        if self.normalize_attn:
            a = F.normalize(a, p=1, dim=(2, 3), eps=1e-12)

        # Multiply attention weights with l to get the attended features
        g = torch.mul(a.expand_as(l), l)  # N x C x W x H

        # Adaptive average pooling to collapse spatial dimensions
        g = F.adaptive_avg_pool2d(g, (1, 1)).view(N, C)  # N x C

        return c.view(N, 1, W, H), g


In [4]:
## Please DONOT remove/change these lines.
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(0)
att = LinearAttentionBlock(in_features=16, normalize_attn=True)

l = torch.rand(4, 16, 64, 64)
g = torch.rand(4, 16, 64, 64)
## Please DONOT remove/change these lines.


### YOUR CODE STARTS HERE ###
#pass the l and g vectors through the attention instance
d, f = att(l, g)
#take the mean of all the elements present in the output 'g' vector and mark your answer for question 2
#hint: use torch.mean()
val = torch.mean(f)
print(val)
### YOUR CODE ENDS HERE ###

RuntimeError: The expanded size of the tensor (1) must match the existing size (16) at non-singleton dimension 1.  Target sizes: [4, 1, 64, 64].  Tensor sizes: [4, 16, 64, 64]

In [1]:
import torch
import numpy as np
import torch.nn.functional as F
import torch.nn as nn

# Define the ConvBlock class
class ConvBlock(nn.Module):
    def __init__(self, in_features, out_features, num_conv, pool=False):
        super(ConvBlock, self).__init__()
        features = [in_features] + [out_features for i in range(num_conv)]

        layers = []
        for i in range(len(features)-1):
            layers.append(nn.Conv2d(in_channels=features[i], out_channels=features[i+1], kernel_size=3, padding=1, bias=True))
            layers.append(nn.BatchNorm2d(num_features=features[i+1], affine=True, track_running_stats=True))
            layers.append(nn.ReLU())
            if pool:
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2, padding=0))
        self.op = nn.Sequential(*layers)

    def forward(self, x):
        return self.op(x)

# Define the ProjectorBlock class
class ProjectorBlock(nn.Module):
    def __init__(self, in_features, out_features):
        super(ProjectorBlock, self).__init__()
        self.op = nn.Conv2d(in_channels=in_features, out_channels=out_features, kernel_size=1, padding=0, bias=False)

    def forward(self, inputs):
        return self.op(inputs)

# Define the LinearAttentionBlock class
class LinearAttentionBlock(nn.Module):
    def __init__(self, in_features, normalize_attn=True):
        super(LinearAttentionBlock, self).__init__()

        self.normalize_attn = normalize_attn
        self.op = nn.Conv2d(in_channels=in_features * 2, out_channels=1, kernel_size=1, padding=0, bias=False)

    def forward(self, l, g):
        N, C, W, H = l.size()

        g_expanded = g.expand(N, C, W, H)

        c = torch.cat((l, g_expanded), dim=1)
        c = self.op(c)

        a = torch.sigmoid(c)

        if self.normalize_attn:
            a = F.normalize(a, p=1, dim=(2, 3), eps=1e-12)

        g_attended = torch.mul(a.expand_as(l), l)
        g_attended = F.adaptive_avg_pool2d(g_attended, (1, 1)).view(N, C)

        return c.view(N, 1, W, H), g_attended

# Setting seeds for reproducibility
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(0)

# Creating an instance of LinearAttentionBlock
att = LinearAttentionBlock(in_features=16, normalize_attn=True)

# Creating random input tensors l and g
l = torch.rand(4, 16, 64, 64)
g = torch.rand(4, 16, 64, 64)

# Passing l and g through the attention block
d, f = att(l, g)

# Calculating the mean of all elements in the tensor f (output of attention block)
val = torch.mean(f)
print(val)


tensor(0.0001, grad_fn=<MeanBackward0>)


### Question 3: Training the Attention VGGNet
Follow the steps in the below given code blocks to build and train the Attention VGGNet model on CIFAR100 dataset.

What is the range of the test set accuracy obtained after training the model for 5 epochs?


1.   0 - 2%
2.   2 - 10%
3.   40 - 50%
4.   25 - 30%


In [3]:
#building a VGGNet integrated with the attention implemented in the above code block
class AttnVGG_before(nn.Module):
    def __init__(self, im_size, num_classes, attention=True, normalize_attn=True):
        super(AttnVGG_before, self).__init__()
        self.attention = attention
        # defining conv blocks with certain choices of input and output features
        #this set of choices is taking from the VGG style model
        self.conv_block1 = ConvBlock(3, 64, 2)
        self.conv_block2 = ConvBlock(64, 128, 2)
        self.conv_block3 = ConvBlock(128, 256, 3)
        self.conv_block4 = ConvBlock(256, 512, 3)
        self.conv_block5 = ConvBlock(512, 512, 3)
        self.conv_block6 = ConvBlock(512, 512, 2, pool=True)
        self.dense = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=int(im_size/32), padding=0, bias=True)

        # defining the projector layers and the multiple attention blocks as mentioned in the paper
        if self.attention:
            self.projector = ProjectorBlock(256, 512)
            self.attn1 = LinearAttentionBlock(in_features=512, normalize_attn=normalize_attn)
            self.attn2 = LinearAttentionBlock(in_features=512, normalize_attn=normalize_attn)
            self.attn3 = LinearAttentionBlock(in_features=512, normalize_attn=normalize_attn)

        # final classification layer
        if self.attention:
            self.classify = nn.Linear(in_features=512*3, out_features=num_classes, bias=True)
        else:
            self.classify = nn.Linear(in_features=512, out_features=num_classes, bias=True)

    def forward(self, x):

        ### YOUR CODE STARTS HERE ###

        # passing the input through all the defined layers
        #pass x through conv_block1 and collect the output in variable name x
        x = self.conv_block1(x)

        #pass x through conv_block2 and collect the output in variable name x
        x = self.conv_block2(x)

        #pass x through conv_block3 and collect the output in variable name l1
        l1 = self.conv_block3(x)

        #pass l1 through max pool 2d and collect the output in variable name x
        x = nn.MaxPool2d(kernel_size=2, stride = 2)(l1)

        #pass x through conv_block4 and collect the output in av ariable called l2
        l2 = self.conv_block4(x)

        #pass l2 through max pool 2d and collect the output in a variable called x
        x = nn.MaxPool2d(kernel_size=2, stride = 2)(l2)

        #pass x through conv_block5 and collect the output in a variable called l3
        l3 = self.conv_block5(x)

       #pass l3 through max pool 2d and collect the output in variable name x
        x = nn.MaxPool2d(kernel_size=2, stride = 2)(l3)

        #pass x through conv_block6 and collect the output in a variable called x
        x = self.conv_block6(x)

        #pass x through self.dense and collect the output in a variable called g
        g = self.dense(x)

        # attention calculations are carried out according to the paper
        if self.attention:

            #pass l1 through self.projector and collect the output in variable t
            t = self.projector(l1)
            #pass g and t through self.attn1 and collect the output in variable names c1 and g1 respectively
            c1, g1 = self.attn1(t,g)

            #pass l2 and g through self.attn2 and collect the output in variable names c2 and g2 respectively
            c2, g2 = self.attn2(l2,g)

            #pass l3 and g through self.attn3 and collect the output in variable names c3 and g3 respectively
            c3, g3 = self.attn3(l3,g)

            #concat g1, g2 and g3 across dim=1 and collect the output in a variable called g
            #hint:use torch.cat()
            g = torch.cat((g1, g2, g3),dim=1)

            # pass g through self.classify layer and collect the output in a variable called x
            x = self.classify(torch.squeeze(g))

        ### YOUR CODE ENDS HERE ###

        #if attention is set to false
        else:
            c1, c2, c3 = None, None, None
            x = self.classify(torch.squeeze(g))

        return [x, c1, c2, c3]

In [4]:
#defining arg parse and running it
parser = argparse.ArgumentParser()

parser.add_argument("--batch_size", type=int, default=128, help="batch size")
parser.add_argument("--epochs", type=int, default=5, help="number of epochs")
parser.add_argument("--lr", type=float, default=0.1, help="initial learning rate")
parser.add_argument("-f", "--file", required=False)


opt = parser.parse_args()

In [5]:
#defining the main function for training
def main():
    print('\nloading the dataset ...\n')

    #setting the image size to 32 as we will be using CIFAR100 dataset
    im_size = 32

    #defining train transforms
    transform_train = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
    ])

    #defining the trainset from the torchvision library
    trainset = torchvision.datasets.CIFAR100(root='CIFAR100_data', train=True, download=True, transform=transform_train)

    #we will be training on a random 5000 image subset of the training data to save time
    sub_train_size = 5000
    train_size = len(trainset) - sub_train_size
    extra_ds, train_ds = random_split(trainset, [train_size, sub_train_size], generator=torch.Generator().manual_seed(30))

    #defining the dataloader
    trainloader = torch.utils.data.DataLoader(train_ds, batch_size=opt.batch_size, shuffle=False, num_workers=0, worker_init_fn = lambda id: np.random.seed(id))
    # testset = torchvision.datasets.CIFAR100(root='CIFAR100_data', train=False, download=False, transform=transform_test)
    # testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=0, worker_init_fn = lambda id: np.random.seed(id))
    print('done')

    print('\npay attention before maxpooling layers...\n')
    #calling the model that needs to be trained
    net = AttnVGG_before(im_size=im_size, num_classes=100,
        attention=True, normalize_attn=True)

    #setting the loss function to CE loss
    criterion = nn.CrossEntropyLoss()
    print('done')

    #setting the device to GPU and transfering model and loss function tot he device
    print('\nmoving to GPU ...\n')
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = net.to(device)
    criterion.to(device)
    print('done')

    # defining the optimizer
    optimizer = optim.SGD(model.parameters(), lr=opt.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.95)

    # training loop starts here
    print('\nstart training ...\n')
    step = 0

    #looping over the total number of epochs
    for epoch in range(opt.epochs):

        # adjusting the learning rate
        scheduler.step()

        print("\nepoch %d learning rate %f\n" % (epoch, optimizer.param_groups[0]['lr']))

        # looping for every epoch
        for i, data in enumerate(trainloader, 0):
            model.train()
            model.zero_grad()
            optimizer.zero_grad()
            inputs, labels = data

            #transfering the data to device
            inputs, labels = inputs.to(device), labels.to(device)

            # forward passing the inputs to the model
            pred, __, __, __ = model(inputs)

            # backwardpass for loss calculation
            loss = criterion(pred, labels)
            loss.backward()
            optimizer.step()

            # display results for every 3 iterations
            if i % 3 == 0:
                model.eval()
                pred, __, __, __ = model(inputs)
                predict = torch.argmax(pred, 1)
                total = labels.size(0)
                correct = torch.eq(predict, labels).sum().double().item()
                accuracy = correct / total
                print("[epoch %d][%d/%d] loss %.4f accuracy %.2f%%"
                    % (epoch, i, len(trainloader)-1, loss.item(), (100*accuracy)))
            step += 1

    #saving the weights of the model at the end of total number of epochs
    torch.save(model.state_dict(), 'net.pth')

In [6]:
#please change the runtime type to GPU before running this cell
#follow the instructions given in this blog for more details: https://www.tutorialspoint.com/google_colab/google_colab_using_free_gpu.htm
main()


loading the dataset ...



100%|██████████| 169001437/169001437 [00:04<00:00, 37486801.53it/s]


Extracting CIFAR100_data/cifar-100-python.tar.gz to CIFAR100_data
done

pay attention before maxpooling layers...

done

moving to GPU ...

done

start training ...


epoch 0 learning rate 0.095000



/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


[epoch 0][0/39] loss 4.6071 accuracy 2.34%
[epoch 0][3/39] loss 4.6057 accuracy 1.56%


KeyboardInterrupt: 

In [19]:
transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
    ])
testset = torchvision.datasets.CIFAR100(root='CIFAR100_data', train=False, download=False, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=0, worker_init_fn = lambda id: np.random.seed(id))
model = AttnVGG_before(im_size=32, num_classes=100, attention=True, normalize_attn=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load('/content/net.pth', map_location=torch.device("cuda")))
model.to(device)
model.eval()
total = 0
correct = 0
with torch.no_grad():
    for i, data in enumerate(testloader, 0):
        images_test, labels_test = data
        images_test, labels_test = images_test.to(device), labels_test.to(device)
        pred_test, __, __, __ = model(images_test)
        predict = torch.argmax(pred_test, 1)
        total += labels_test.size(0)
        correct += torch.eq(predict, labels_test).sum().double().item()
    print("\nAccuracy on test data: %.2f%%\n" % (100*correct/total))

FileNotFoundError: [Errno 2] No such file or directory: '/content/net.pth'